# Scientific computing in the browser

This notebook shows a typical scientific Python workflow running entirely in
your browser via WebAssembly — no server required.

We'll install [SciPy](https://scipy.org/) at runtime with `%conda install`,
then use it for signal processing, linear algebra, and polynomial fitting.

*Examples adapted from the
[SciPy documentation](https://docs.scipy.org/doc/scipy/tutorial/signal.html)
(BSD 3-Clause license) and
[NumPy documentation](https://numpy.org/doc/stable/reference/routines.fft.html)
(BSD 3-Clause license).*

## Install SciPy

SciPy is a large scientific computing library with compiled Fortran/C code
(LAPACK, BLAS, signal processing routines). Install it with one command:

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install scipy

## Signal processing with SciPy

Now that SciPy is installed, let's use it. Generate a noisy signal, apply a
Butterworth low-pass filter, and visualize the result — a classic DSP workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng(42)
fs = 1000  # sampling rate (Hz)
t = np.arange(0, 1.0, 1.0 / fs)

# Two sine waves buried in noise
clean = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
noisy = clean + 0.8 * rng.standard_normal(len(t))

# 4th-order Butterworth low-pass at 20 Hz
sos = signal.butter(4, 20, btype="low", fs=fs, output="sos")
filtered = signal.sosfilt(sos, noisy)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, data, title, color in zip(
    axes,
    [clean, noisy, filtered],
    ["Original signal (5 Hz + 12 Hz)", "With noise", "After Butterworth low-pass (20 Hz cutoff)"],
    ["steelblue", "gray", "tomato"],
):
    ax.plot(t[:200], data[:200], color=color, linewidth=1.2)
    ax.set_title(title)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Signal filtering — SciPy in WebAssembly", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Linear algebra

Matrix operations using pure NumPy — vectorized arithmetic compiled to WASM.

In [ ]:
A = rng.standard_normal((5, 5))
A = A + A.T  # make symmetric

# Power iteration: find the dominant eigenvalue without LAPACK
v = rng.standard_normal(A.shape[0])
v = v / np.linalg.norm(v)

for step in range(50):
    w = A @ v
    eigenvalue = v @ w
    v = w / np.linalg.norm(w)

print(f"Matrix shape:       {A.shape}")
print(f"Dominant eigenvalue: {eigenvalue:+.6f}")
print(f"Rayleigh residual:   {np.linalg.norm(A @ v - eigenvalue * v):.2e}")

# Matrix properties via pure operations
print(f"\nTrace (sum of diag):  {np.trace(A):+.6f}")
print(f"Frobenius norm:       {np.sqrt(np.sum(A**2)):.6f}")
print(f"Symmetric check:      {np.allclose(A, A.T)}")

## Polynomial fitting

Fit a polynomial to noisy data using NumPy — the same Vandermonde matrix
approach you'd use on your laptop.

In [ ]:
t_data = np.linspace(0, 2, 200)
y_true = np.sin(2 * np.pi * t_data) * np.exp(-t_data)
y_noisy = y_true + 0.15 * rng.standard_normal(len(t_data))

# Fit polynomials of increasing degree
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(t_data[::3], y_noisy[::3], s=10, alpha=0.5, label="Noisy data", color="gray")
ax.plot(t_data, y_true, "--", color="steelblue", linewidth=1.5, label="True signal")

colors = ["#22c55e", "#f59e0b", "#ef4444"]
for degree, color in zip([3, 7, 15], colors):
    coeffs = np.polyfit(t_data, y_noisy, degree)
    y_fit = np.polyval(coeffs, t_data)
    residual = np.sqrt(np.mean((y_noisy - y_fit) ** 2))
    ax.plot(t_data, y_fit, color=color, linewidth=1.5,
            label=f"Degree {degree} (RMSE={residual:.3f})")

ax.legend(fontsize=9)
ax.set(xlabel="Time (s)", ylabel="Amplitude",
       title="Polynomial fitting — underfitting vs overfitting")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## What just happened

You installed SciPy — a large scientific library with compiled Fortran/C
code — at runtime in your browser, then used it for signal processing,
linear algebra, and polynomial fitting.

Browse available packages at [emscripten-forge.org](https://emscripten-forge.org/)
or use `%conda search <package>` to explore from this notebook.